In [ ]:
!pip install --upgrade numpy --target ./python
!pip install --upgrade numexpr --target ./python

In [ ]:
import sys
sys.path.append(r"./python")

import os
import json
from model import *

#根据时间情况修改index和language值
index =  "singtao_demo"

embedding_endpoint_name = "cohere.embed-multilingual-v3"

embedding_type = 'bedrock' if embedding_endpoint_name.find('titan') or embedding_endpoint_name.find('cohere') else 'sagemaker'
embeddings = init_embeddings_bedrock(embedding_endpoint_name)

In [ ]:
import sys
sys.path.append(r"./python")

from tqdm import tqdm
from opensearch_multimodel_dataload import add_multimodel_documents
import re
import io
import time
import csv
text_max_length = 2000

files_path = '../docs/singtao/'
files = os.listdir(files_path)
for file in files:
    file_path = files_path + file
    print('file_path:',file_path)
    
    texts = []
    metadatas = []
    
    i = 0
    with open(file_path, newline='') as csvfile:
        reader = csv.reader(csvfile)

        for row in reader:
            i += 1
            if i == 1:
                continue
            print('i:',i)
            news_id = row[0]
            category = row[1]
            publish_datatime = row[2]
            title = row[3]
            html = row[4]
            thumbnail,description = getContentInfo(html)
            paragraph = '<title>' + title + '</title>'+'<description>' + description + '</description>'
            texts.append(paragraph)
            metadata = {}
            metadata['sentence'] = title[:text_max_length] if len(title) > text_max_length else title
            metadata['thumbnail'] = thumbnail
            metadata['description'] = description
            metadatas.append(metadata)

        if len(texts) > 0:
            if embedding_type == 'bedrock':
                text_embeddings = embeddings.embed_documents([metadata['sentence'] for metadata in metadatas])
            else:
                text_embeddings = embeddings.embed_documents([metadata['sentence'] for metadata in metadatas],chunk_size=10)

            print('texts len:',len(texts))
            print('metadatas len:',len(metadatas))
            print('embeddings len:',len(text_embeddings))
            print('begin to save in vectore store')

            add_multimodel_documents(
                index,
                texts=texts,
                embeddings=text_embeddings,
                metadatas=metadatas
            )
            print('finish save in vectore store:',index)

